In [6]:
from pathlib import Path

files = sorted(Path("../data/models").glob("*/*/*.json"))

print("Number of files:", len(files))
print(*files[:10], sep="\n")

Number of files: 9611
../data/models/gemma-3n-e4b-it/objective_energy/manifest.json
../data/models/gemma-3n-e4b-it/objective_energy/math_test_101__J0__rep00.json
../data/models/gemma-3n-e4b-it/objective_energy/math_test_101__J0__rep01.json
../data/models/gemma-3n-e4b-it/objective_energy/math_test_101__J0__rep02.json
../data/models/gemma-3n-e4b-it/objective_energy/math_test_101__J0__rep03.json
../data/models/gemma-3n-e4b-it/objective_energy/math_test_101__J1__rep00.json
../data/models/gemma-3n-e4b-it/objective_energy/math_test_101__J1__rep01.json
../data/models/gemma-3n-e4b-it/objective_energy/math_test_101__J1__rep02.json
../data/models/gemma-3n-e4b-it/objective_energy/math_test_101__J1__rep03.json
../data/models/gemma-3n-e4b-it/objective_energy/math_test_101__J2__rep00.json


In [7]:
import json, glob
from pathlib import Path
import pandas as pd

records = []
for f in sorted(Path("../data/models").glob("*/*/*.json")):
    if f.name == "manifest.json":
        continue
    rec = json.load(open(f))
    rec["model_dir"] = f.parts[-3]   # e.g. gemma-3n-e4b-it
    rec["energy_mode"] = f.parts[-2] # objective_energy / subjective_energy
    rec["file"] = f.name
    records.append(rec)

df = pd.json_normalize(records)

parts = df["file"].str.extract(r"(?P<question>.+)__(?P<graph>[^_]+)__rep(?P<rep>\d+)\.json")
df = df.join(parts)
df["replica"] = df["rep"].astype(int)

df["model"] = df["meta.model"]

# load them and add to the df here
obj = pd.concat([
    pd.read_json("../data/obj/train.jsonl", lines=True),
    pd.read_json("../data/obj/test.jsonl", lines=True),
])
ground_truth = obj.set_index("qid")["answer"]              # "A" or "B"
lean = json.load(open("../data/subj/lean.json"))["lean"]   # qid -> left/right/ambiguous

df["ground_truth"] = df["question"].map(ground_truth)      # NaN on subjective rows
df["political_lean"] = df["question"].map(lean)            # NaN on objective rows

# sanity check: every row got exactly one of the two labels
assert (df["ground_truth"].notna() ^ df["political_lean"].notna()).all()
cols = ['model', 'mode', 'statement', 'J', 'spins_history', 'spins_raw_history',
        'replica', 'ground_truth', 'political_lean']
df[cols].to_json("../data/clean_runs.json", orient="records")

# from datasets import Dataset
# ds = Dataset.from_pandas(df[cols].reset_index(drop=True))
# ds.push_to_hub("physics-of-agents/agent-opinions", private=False)

In [8]:
from pathlib import Path

p = Path("../data/clean_runs.json")
print(p.resolve())
print(p.exists())
print(p.stat().st_size if p.exists() else None)

/Users/leo/Documents/github/physics-of-agents/data/clean_runs.json
True
73214887
